# Ekstraksi Fitur Audio Menggunakan Librosa
## Proyek Sains Data (PSD) - Klasifikasi Audio: SAWMIL vs NON SAWMIL

Notebook ini digunakan untuk mengekstraksi fitur-fitur audio representatif dari kumpulan audio **SAWMIL** dan **NON SAWMIL** di dalam folder `PSD KLASIFIKASI AUDIO`.

Fitur yang diekstraksi meliputi:
- **MFCC (Mel-Frequency Cepstral Coefficients)** (20 koefisien: Mean & Standar Deviasi)
- **Chroma STFT** (12 bin: Mean & Standar Deviasi)
- **Spectral Centroid** (Mean & Standar Deviasi)
- **Spectral Bandwidth** (Mean & Standar Deviasi)
- **Spectral Rolloff** (Mean & Standar Deviasi)
- **Zero Crossing Rate (ZCR)** (Mean & Standar Deviasi)
- **Root Mean Square (RMS) Energy** (Mean & Standar Deviasi)

Hasil ekstraksi akan disimpan dalam format tabular (`pandas.DataFrame`) dan diekspor ke file CSV (`hasil_ekstraksi_audio.csv`) sehingga siap digunakan untuk pemodelan Machine Learning.

### 2. Import Library yang Dibutuhkan

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

### 3. Fungsi Ekstraksi Fitur Audio per File

In [2]:
def extract_features(file_path, sr=22050, n_mfcc=20):
    """
    Mengekstrak kumpulan fitur audio numerik dari satu file audio menggunakan librosa.
    """
    try:
        # Load file audio (resample ke sample rate tertentu)
        y, sr = librosa.load(file_path, sr=sr)
        
        # Inisialisasi dictionary penyimpan fitur
        features = {}
        
        # 1. MFCC (20 Koefisien)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        for i in range(n_mfcc):
            features[f'mfcc_{i+1}_mean'] = float(np.mean(mfcc[i]))
            features[f'mfcc_{i+1}_std'] = float(np.std(mfcc[i]))
            
        # 2. Chroma STFT (12 Pitch Class)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(chroma.shape[0]):
            features[f'chroma_{i+1}_mean'] = float(np.mean(chroma[i]))
            features[f'chroma_{i+1}_std'] = float(np.std(chroma[i]))
            
        # 3. Spectral Centroid
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        features['spectral_centroid_mean'] = float(np.mean(spec_cent))
        features['spectral_centroid_std'] = float(np.std(spec_cent))
        
        # 4. Spectral Bandwidth
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features['spectral_bandwidth_mean'] = float(np.mean(spec_bw))
        features['spectral_bandwidth_std'] = float(np.std(spec_bw))
        
        # 5. Spectral Rolloff
        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features['spectral_rolloff_mean'] = float(np.mean(spec_rolloff))
        features['spectral_rolloff_std'] = float(np.std(spec_rolloff))
        
        # 6. Zero Crossing Rate (ZCR)
        zcr = librosa.feature.zero_crossing_rate(y=y)
        features['zcr_mean'] = float(np.mean(zcr))
        features['zcr_std'] = float(np.std(zcr))
        
        # 7. Root Mean Square (RMS) Energy
        rms = librosa.feature.rms(y=y)
        features['rms_mean'] = float(np.mean(rms))
        features['rms_std'] = float(np.std(rms))
        
        return features
    except Exception as e:
        print(f"Gagal memproses file {file_path}: {e}")
        return None

### 4. Ekstraksi Seluruh File dalam Folder SAWMIL dan NON SAWMIL

In [3]:
# Path dataset induk
dataset_path = 'PSD KLASIFIKASI AUDIO'
classes = ['SAWMIL', 'NON SAWMIL']

all_features = []

for label in classes:
    folder = os.path.join(dataset_path, label)
    if not os.path.exists(folder):
        print(f"Direktori tidak ditemukan: {folder}")
        continue
        
    # Ambil semua file audio mp3/wav
    audio_files = [f for f in os.listdir(folder) if f.lower().endswith(('.mp3', '.wav', '.ogg'))]
    print(f"Memproses kategori '{label}' ({len(audio_files)} audio)...")
    
    for filename in tqdm(audio_files, desc=label):
        file_path = os.path.join(folder, filename)
        feat = extract_features(file_path)
        
        if feat is not None:
            # Tambahkan metadata filename dan label target
            feat['filename'] = filename
            feat['label'] = label
            all_features.append(feat)

# Konversi list of dicts ke DataFrame Pandas
df = pd.DataFrame(all_features)

# Mengatur posisi kolom: filename & label di posisi terdepan
cols = ['filename', 'label'] + [c for c in df.columns if c not in ['filename', 'label']]
df = df[cols]

print("\n--- Proses Ekstraksi Selesai! ---")
print(f"Total baris data: {df.shape[0]}")
print(f"Total fitur yang diekstraksi: {df.shape[1] - 2}")

Memproses kategori 'SAWMIL' (100 audio)...


SAWMIL:   0%|          | 0/100 [00:00<?, ?it/s]

SAWMIL: 100%|██████████| 100/100 [00:35<00:00,  2.81it/s]


Memproses kategori 'NON SAWMIL' (100 audio)...


NON SAWMIL: 100%|██████████| 100/100 [00:14<00:00,  7.10it/s]


--- Proses Ekstraksi Selesai! ---
Total baris data: 200
Total fitur yang diekstraksi: 74


### 5. Menampilkan dan Memeriksa Hasil Ekstraksi

In [4]:
# Menampilkan 5 sampel data teratas
df.head()

,filename,label,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,...,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,zcr_mean,zcr_std,rms_mean,rms_std
0,1.mp3,SAWMIL,-154.092499,23.846697,92.204659,13.963578,49.628963,8.349650,47.014332,6.460558,...,1975.898071,398.667158,2740.698550,279.290362,5164.828491,1027.055430,0.077980,0.018720,0.105194,0.014928
1,10.mp3,SAWMIL,-90.875938,24.508070,58.004902,16.022915,-15.412866,8.375935,-11.365315,7.831729,...,3385.732029,433.215749,3079.102760,80.019106,7855.681356,355.678366,0.153641,0.056451,0.092373,0.020678
2,100.mp3,SAWMIL,-88.144463,30.498030,96.341797,36.586918,-8.203456,14.114124,10.363613,16.264250,...,2198.079427,617.058055,2417.036452,343.385056,4649.427287,1346.482860,0.102141,0.022456,0.142866,0.030866
3,11.mp3,SAWMIL,-202.864624,52.743198,59.629662,17.923719,3.296696,9.415070,12.391150,11.962558,...,3806.184872,696.705519,3189.021224,228.935572,8098.129272,877.778870,0.224453,0.058238,0.026545,0.016727
4,12.mp3,SAWMIL,-98.233940,46.728466,34.001640,13.481186,-14.537036,13.989621,6.683242,9.979526,...,4029.845803,286.560861,2964.657321,209.937052,7714.220174,580.883033,0.266446,0.035347,0.070882,0.024069


In [5]:
# Distribusi jumlah sampel tiap kelas
df['label'].value_counts()

label
SAWMIL        100
NON SAWMIL    100
Name: count, dtype: int64

### 6. Menyimpan Hasil Ekstraksi ke File CSV

In [6]:
output_csv = 'hasil_ekstraksi_audio.csv'
df.to_csv(output_csv, index=False)
print(f"File hasil ekstraksi berhasil disimpan sebagai: '{output_csv}'")

File hasil ekstraksi berhasil disimpan sebagai: 'hasil_ekstraksi_audio.csv'
